<a href="https://colab.research.google.com/github/JZjj/llm-sys-project/blob/main/LLM%20Code%20Generation/Single%20Model/zero_shot_and_few_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library

# Data Loading

In [1]:
import os
import subprocess
from pathlib import Path

CASTLE_REPO = "https://github.com/CASTLE-Benchmark/CASTLE-Benchmark.git"
# Fix: Use a path relative to the current working directory in Colab
DATA_DIR = Path("/content") / "data" / "CASTLE-Benchmark"

def clone_if_needed():
    if DATA_DIR.exists():
        print("CASTLE already cloned at", DATA_DIR)
        return
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", CASTLE_REPO, str(DATA_DIR)], check=True)
    print("Cloned CASTLE to", DATA_DIR)

def list_cases():
    # CASTLE repo structure: each test is under e.g. tests/... (repo may change — inspect after cloning)
    # Here we do a simple walk to collect .c/.h files
    file_list = []
    for p in DATA_DIR.rglob("*.c"):
        file_list.append(str(p))
    for p in DATA_DIR.rglob("*.cpp"):
        file_list.append(str(p))
    print(f"Found {len(file_list)} source files")
    return file_list

if __name__ == "__main__":
    clone_if_needed()
    files = list_cases()
    # Optionally save listing
    out = DATA_DIR.parent / "file_listing.txt"
    out.write_text("\n".join(files))
    print("Wrote listing to", out)


CASTLE already cloned at /content/data/CASTLE-Benchmark
Found 250 source files
Wrote listing to /content/data/file_listing.txt


# LLM Model Selection

# groq_API

- Free plan [rate limits](https://console.groq.com/docs/rate-limits?utm_source=chatgpt.com)

- Model selection

> RPM: Requests per minute
>
> RPD: Requests per day
>
> TPM: Tokens per minute
>
> TPD: Tokens per day

|Model Name |RPM | 	RPD| 	TPM	|TPD| Advantage|
|---|---|---|---|---|---|
|meta-llama/llama-4-scout-17b-16e-instruct|30|1K|30K|500K|Specifically trained for reasoning and code safety awareness|
|openai/gpt-oss-20b|30|1K|8K|200K|High code quality, Strong debugging ability ,Fast reasoning (very fast on Groq)|
|llama-3.1-8b-instant|30|14.4K|6K|500K| Fast|

In [2]:
!pip install groq

In [3]:
# Colab get the corresponding api key from Secret
from google.colab import userdata
api_key=userdata.get("GROQ_API_KEY")

# Select model
Model_name="meta-llama/llama-4-scout-17b-16e-instruct"

# GROQ

from groq import Groq

client = Groq(
    api_key=api_key,
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model=Model_name,
)

print(chat_completion.choices[0].message.content)

Fast language models have become increasingly important in recent years due to their wide range of applications and significant impact on various industries. Here are some reasons highlighting their importance:

1. **Improved User Experience**: Fast language models enable faster response times, which is crucial for applications like chatbots, virtual assistants, and customer service platforms. A responsive interface can make a significant difference in user satisfaction, engagement, and overall experience.

2. **Increased Efficiency**: In applications like language translation, text summarization, and sentiment analysis, speed can directly translate to cost savings and increased productivity. Faster models can process larger volumes of text data in less time, making them more efficient.

3. **Real-time Applications**: Certain applications, such as live subtitles, simultaneous interpretation, and real-time sentiment analysis during events or on social media, require language models to p

# Zero Shot

Ask an LLM to review and refine existing code to strengthen its security posture.

Prompt:

"You are an expert C programmer specializing in secure coding practices.
Please review the following C code and identify any potential security vulnerabilities, such as buffer overflows, format string bugs, integer overflows, memory leaks, or improper input validation.
Rewrite the code to eliminate these vulnerabilities, ensuring the improved code is robust, functional, and follows secure coding best practices.
Only provide the improved C code block, enclosed in triple backticks (```c\n...\n```), and nothing else."


Read all code file paths listed in `file_listing.txt`, send each file's content to the LLM for improvement, and save all improved code outputs to the `improved_c_code` directory.

For testing, we only improve the codes in the first two files.

In [21]:
import os
from pathlib import Path
import re

# -----------------------------
# Extract code from LLM response
# -----------------------------
def extract_code_from_response(text: str, language: str = "c") -> str:
    pattern = rf"```(?:{language}\\n|{language}\\r\\n|{language}\\s*)?([\\s\\S]*?)```"
    m = re.search(pattern, text)
    if m:
        return m.group(1).strip()

    m_general = re.search(r"```(?:\\n|\\r\\n|\\s*)?([\\s\\S]*?)```", text)
    if m_general:
        return m_general.group(1).strip()

    return text.strip()

# -----------------------------
# Load all file paths
# -----------------------------
file_list_path = "/content/data/file_listing.txt"

if not Path(file_list_path).exists():
    raise FileNotFoundError("file_listing.txt not found!")

with open(file_list_path, "r") as f:
    c_files = [line.strip() for line in f if line.strip()]

print(f"Found {len(c_files)} C files to process.")

# -----------------------------
# Where to save improved code
# -----------------------------
output_dir_zero_shot = Path("/content/improved_c_code")
output_dir=output_dir_zero_shot
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Process each file
# -----------------------------
for c_file_path in c_files[:2]:
    print(f"\n==============================")
    print(f"Processing: {c_file_path}")
    print("==============================")

    # 1. Read C code
    c_file_path = Path(c_file_path)
    try:
        c_code_original = c_file_path.read_text()
        print(f"Successfully read: {c_file_path}")
    except Exception as e:
        print(f"Error reading file: {e}")
        continue

    # 2. Construct LLM prompt
    prompt_content = f"""You are an expert C programmer specializing in secure coding practices.
    Please review the following C code and identify any potential security vulnerabilities, such as buffer overflows, format string bugs, integer overflows, memory leaks, or improper input validation.
    Rewrite the code to eliminate these vulnerabilities, ensuring the improved code is robust, functional, and follows secure coding best practices.
    Only provide the improved C code block, enclosed in triple backticks (```c\\n...\\n```), and nothing else.

    ```c
    {c_code_original}
    """
    messages = [
        {"role": "system", "content": "You are a helpful and experienced C programming assistant focused on security improvements."},
        {"role": "user", "content": prompt_content}
    ]

    print("Sending request to LLM...")

    try:
        chat_completion = client.chat.completions.create(
            messages=messages,
            model=Model_name,
            temperature=0.7
        )
        llm_response_content = chat_completion.choices[0].message.content

        # 3. Extract code
        improved_c_code = extract_code_from_response(llm_response_content, language="c")

        # 4. Save result
        out_name = c_file_path.stem + "_improved.c"
        out_path = output_dir / out_name
        out_path.write_text(improved_c_code)

        print(f"Saved improved file to: {out_path}")

    except Exception as e:
        print(f"Error during LLM call: {e}")
        print(f"Raw response: {llm_response_content if 'llm_response_content' in locals() else 'N/A'}")

Found 250 C files to process.

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code/CASTLE-415-1_improved.c

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code/CASTLE-843-2_improved.c


In [22]:
# Print the original code and the improved code
from pathlib import Path

c_file_path = Path(c_files[1]) # Convert to Path object here
c_code_original = c_file_path.read_text()

out_name = c_file_path.stem + "_improved.c"
out_path = output_dir_zero_shot / out_name
improved_c_code = Path(out_path).read_text()
print("\n--- Original C Code ---")
print(c_code_original)
print("\n--- Improved C Code in zero-shot mode ---")
print(improved_c_code)


--- Original C Code ---
#include <stdio.h>
#include <string.h>

void printNum(int* num) {
    printf("%d\n", *num);
}

int main() {
    int a = 55;
    float b = 1.f;

    printNum(&a);
    printNum(&b);

    return 0;
}

--- Improved C Code in zero-shot mode ---
```c
#include <stdio.h>
#include <stdint.h>

void printNum(int32_t num) {
    printf("%d\n", num);
}

int main() {
    int32_t a = 55;

    printNum(a);

    return 0;
}
```


# Few-Shot

Few shot examples

In [6]:
few_shot_examples = """
Here are examples of how insecure code should be rewritten securely:

Example 1:
Input Code:
```c
char buf[10];
gets(buf);   // unsafe
```

Improved Code:
```c
char buf[10];
fgets(buf, sizeof(buf), stdin);   // safe
```

Example 2:
Input Code:
```c
int arr[5];
for (int i = 0; i <= 5; i++) {   // off-by-one
    printf("%d", arr[i]);
}
```
Improved Code:
```c
int arr[5];
for (int i = 0; i < 5; i++) {
    printf("%d", arr[i]);
}
```
"""

Call LLM

In [17]:
import os
from pathlib import Path
import re

# -----------------------------
# Extract code from LLM response
# -----------------------------
def extract_code_from_response(text: str, language: str = "c") -> str:
    pattern = rf"```(?:{language}\\n|{language}\\r\\n|{language}\\s*)?([\\s\\S]*?)```"
    m = re.search(pattern, text)
    if m:
        return m.group(1).strip()

    m_general = re.search(r"```(?:\\n|\\r\\n|\\s*)?([\\s\\S]*?)```", text)
    if m_general:
        return m_general.group(1).strip()

    return text.strip()

# -----------------------------
# Load all file paths
# -----------------------------
file_list_path = "/content/data/file_listing.txt"

if not Path(file_list_path).exists():
    raise FileNotFoundError("file_listing.txt not found!")

with open(file_list_path, "r") as f:
    c_files = [line.strip() for line in f if line.strip()]

print(f"Found {len(c_files)} C files to process.")

# -----------------------------
# Where to save improved code
# -----------------------------
output_dir_few_shot = Path("/content/improved_c_code_few_shot")
output_dir=output_dir_few_shot
output_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Process each file
# -----------------------------
for c_file_path in c_files[:2]:
    print(f"\n==============================")
    print(f"Processing: {c_file_path}")
    print("==============================")

    # 1. Read C code
    c_file_path = Path(c_file_path)
    try:
        c_code_original = c_file_path.read_text()
        print(f"Successfully read: {c_file_path}")
    except Exception as e:
        print(f"Error reading file: {e}")
        continue

    # 2. Construct LLM prompt
    prompt_content = f"""
    You are an expert C programmer specializing in secure coding practices.

    Below are examples of insecure C code and their corrected secure versions.
    Use them as a reference for style and security rewriting.

      {few_shot_examples}

    Now improve the following C code securely.
    Return only the improved C code in a ```c ... ``` block.

    Input Code:
    ```c
    {c_code_original}
    ```
    """
    messages = [
        {"role": "system", "content": "You are a helpful and experienced C programming assistant focused on security improvements."},
        {"role": "user", "content": prompt_content}
    ]

    print("Sending request to LLM...")

    try:
        chat_completion = client.chat.completions.create(
            messages=messages,
            model=Model_name,
            temperature=0.7
        )
        llm_response_content = chat_completion.choices[0].message.content

        # 3. Extract code
        improved_c_code = extract_code_from_response(llm_response_content, language="c")

        # 4. Save result
        out_name = c_file_path.stem + "_few_shot_improved.c"
        out_path = output_dir / out_name
        out_path.write_text(improved_c_code)

        print(f"Saved improved file to: {out_path}")

    except Exception as e:
        print(f"Error during LLM call: {e}")
        print(f"Raw response: {llm_response_content if 'llm_response_content' in locals() else 'N/A'}")

Found 250 C files to process.

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-415-1.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code_few_shot/CASTLE-415-1_few_shot_improved.c

Processing: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Successfully read: /content/data/CASTLE-Benchmark/datasets/CASTLE-C250/CASTLE-843-2.c
Sending request to LLM...
Saved improved file to: /content/improved_c_code_few_shot/CASTLE-843-2_few_shot_improved.c


Print the original code and the improved code.

In [19]:
from pathlib import Path

c_file_path = Path(c_files[1]) # Convert to Path object here
c_code_original = c_file_path.read_text()

out_name_few_shot = c_file_path.stem + "_few_shot_improved.c"
out_path = output_dir_few_shot / out_name_few_shot
improved_c_code_few_shot = Path(out_path).read_text()
print("\n--- Original C Code ---")
print(c_code_original)
print("\n--- Improved C Code in few-shot mode ---")
print(improved_c_code_few_shot)


--- Original C Code ---
#include <stdio.h>
#include <string.h>

void printNum(int* num) {
    printf("%d\n", *num);
}

int main() {
    int a = 55;
    float b = 1.f;

    printNum(&a);
    printNum(&b);

    return 0;
}

--- Improved C Code in few-shot mode ---
```c
#include <stdio.h>

void printNum(int* num) {
    if (num != NULL) {
        printf("%d\n", *num);
    } else {
        fprintf(stderr, "Error: NULL pointer passed to printNum\n");
    }
}

int main() {
    int a = 55;

    printNum(&a);

    // Avoid passing float pointer directly
    int c = (int)1;
    // Or use a union or a float specific function

    return 0;
}
```


# Evaluation-metric-based prompting